In [4]:
"""Builds extract_addresses.ipynb — the one-time payout-address extraction
(cost-controlled, year-by-year, resumable) plus assembly into the full
two-method attribution that unlocks the H1-H5 heuristic axis."""
import nbformat as nbf

nb = nbf.v4.new_notebook()
c = []
md = lambda s: c.append(nbf.v4.new_markdown_cell(s))
code = lambda s: c.append(nbf.v4.new_code_cell(s))




import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

try:
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame
    from stage2_attribute.reference_loader import load_reference
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, os.path.abspath("../src"))
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame
    from stage2_attribute.reference_loader import load_reference

PROJECT_ID = "project-9536b219-93fb-4fca-aac"   # <-- same ID as before
client = bigquery.Client(project=PROJECT_ID)

def year_query(year):
    """Coinbase payout addresses for one year (only addresses; pruned by time)."""
    return f"""
    SELECT
      block_number AS height,
      ARRAY(SELECT addr FROM UNNEST(outputs) AS o, UNNEST(o.addresses) AS addr) AS output_addresses
    FROM `bigquery-public-data.crypto_bitcoin.transactions`
    WHERE is_coinbase = TRUE
      AND block_timestamp >= TIMESTAMP('{year}-01-01')
      AND block_timestamp <  TIMESTAMP('{year + 1}-01-01')
    """

YEARS = list(range(2009, 2027))
print("ready")

ready


In [5]:
costs = {}
for y in YEARS:
    dry = client.query(year_query(y), bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    costs[y] = dry.total_bytes_processed / 1e9
    print(f"  {y}:  {costs[y]:8.2f} GB")

total = sum(costs.values())
print("-" * 28)
print(f"  TOTAL: {total:8.1f} GB   ({total/1000:.2f} TB)")
print(f"\\nFree tier covers the first 1000 GB this month.")
if total > 1500:
    print("** Over ~1.5 TB — consider pulling recent years only, or ask about node exports. **")


  2009:    183.21 GB
  2010:    183.21 GB
  2011:    183.21 GB
  2012:    183.21 GB
  2013:    183.21 GB
  2014:    183.21 GB
  2015:    183.21 GB
  2016:    183.21 GB
  2017:    183.21 GB
  2018:    183.21 GB
  2019:    183.21 GB
  2020:    183.21 GB
  2021:    183.21 GB
  2022:    183.21 GB
  2023:    183.21 GB
  2024:    183.21 GB
  2025:    183.21 GB
  2026:    183.21 GB
----------------------------
  TOTAL:   3297.8 GB   (3.30 TB)
\nFree tier covers the first 1000 GB this month.
** Over ~1.5 TB — consider pulling recent years only, or ask about node exports. **
